# PowerShell Tips and Tricks

Author: Eric K. Miller

Last updated: 11 June 2026

This notebook contains PowerShell snippets to understand and use the environment for automating Windows tasks. Since these scripts are run in a Polyglot Notebook, the code is specific to **PowerShell Core version 7.x.**

## Environment/System

In [1]:
# Get PowerShell version information
$PSVersionTable | Format-Table -AutoSize | Out-Default


Name                      Value
----                      -----
PSVersion                 7.5.1
PSEdition                 Core
GitCommitId               7.5.1
OS                        Microsoft Windows 10.0.26200
Platform                  Win32NT
PSCompatibleVersions      {1.0, 2.0, 3.0, 4.0…}
PSRemotingProtocolVersion 2.3
SerializationVersion      1.1.0.1
WSManStackVersion         3.0



In [2]:
# Get just the PowerShell version
# Uses an alias for Format-Table, ft
$PSVersionTable.PSVersion | ft | Out-Default


Major  Minor  Patch  PreReleaseLabel BuildLabel
-----  -----  -----  --------------- ----------
7      5      1                      



In [3]:
# Get .NET version (.NET Standard 2.0 supports both v5.1 and v7.x)
[System.Runtime.InteropServices.RuntimeInformation]::FrameworkDescription

.NET 9.0.12


May need to enable running scripts. For more information, see [https:/go.microsoft.com/fwlink/?LinkID=135170](https:/go.microsoft.com/fwlink/?LinkID=135170):

`Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope LocalMachine`

In ISE/Console Only:

- Update the prompt:
`function prompt {"ps4ds > "}`

- Run as administrator -> `Update-Help`

Create dialog boxes (*in the Console/Terminal only*)

In [ ]:
# Enable the capability (Assembly)
Add-Type -AssemblyName System.Windows.Forms

$Form = New-Object System.Windows.Forms.Form

$OpenFileDialog = New-Object System.Windows.Forms.OpenFileDialog  # and SaveFileDialog
# OR [System.Windows.Forms.OpenFileDialog]::new()

$OpenFileDialog.Filter = "Word Documents (*.docx) | *.docx"

$OpenFileDialog.ShowDialog() | Out-Null
# OR $null = $OpenFileDialog.ShowDialog()
# OR [void]$OpenFileDialog.ShowDialog()

$WordFile = $OpenFileDialog.FileName

## Syntax/Setup

In [5]:
# Get cmdlet aliases
Get-Alias | Select-Object Name, Definition -First 10 | Format-Table | Out-Default


Name  Definition
----  ----------
?     Where-Object
%     ForEach-Object
ac    Add-Content
cat   Get-Content
cd    Set-Location
chdir Set-Location
clc   Clear-Content
clear Clear-Host
clhy  Clear-History
cli   Clear-Item



In [6]:
# Get specific cmdlet alias(es)
Get-Alias -Definition Get-ChildItem | ft | Out-Default


CommandType     Name                                               Version    Source
-----------     ----                                               -------    ------
Alias           dir -> Get-ChildItem                                          
Alias           gci -> Get-ChildItem                                          
Alias           ls -> Get-ChildItem                                           



To use environment variables, it is best to store them in a user profile.

If the profile does not exist, you may have to create it in ...\Documents\PowerShell
- To see whether it exists: `Test-Path $profile`
- To see where it exists: `$profile`

After creating the directory (if necessary), access the profile file by calling `notepad $profile`
- Set environment variables as variables
- Store common start-up settings
- To reload the profile and use the variables immediately: `. $profile` (dot-sourcing)

It loads each time PowerShell loads.

In [7]:
# Obfuscate user/system-specific filepaths
$ppath = '...' + ($PROFILE -split [regex]::Escape($env:USERNAME))[1]
Write-Host "Profile saved at $ppath"

Profile saved at ...\OneDrive\Documents\PowerShell\Microsoft.dotnet-interactive_profile.ps1


### Functions

#### Function Providers

In [10]:
# Uses aliases for Select-Object, Format-Table
Get-ChildItem Function: | select Name -First 10 | ft | Out-Default


Name
----
A:
B:
C:
cd..
cd\
cd~
Clear-Host
D:
E:
F:



In [11]:
# Get the function definition
(Get-ChildItem Function:mkdir).Definition


<#
.FORWARDHELPTARGETNAME New-Item
.FORWARDHELPCATEGORY Cmdlet
#>

[CmdletBinding(DefaultParameterSetName='pathSet',
    SupportsShouldProcess=$true,
    SupportsTransactions=$true,
    ConfirmImpact='Medium')]
    [OutputType([System.IO.DirectoryInfo])]
param(
    [Parameter(ParameterSetName='nameSet', Position=0, ValueFromPipelineByPropertyName=$true)]
    [Parameter(ParameterSetName='pathSet', Mandatory=$true, Position=0, ValueFromPipelineByPropertyName=$true)]
    [System.String[]]
    ${Path},

    [Parameter(ParameterSetName='nameSet', Mandatory=$true, ValueFromPipelineByPropertyName=$true)]
    [AllowNull()]
    [AllowEmptyString()]
    [System.String]
    ${Name},

    [Parameter(ValueFromPipeline=$true, ValueFromPipelineByPropertyName=$true)]
    [System.Object]
    ${Value},

    [Switch]
    ${Force},

    [Parameter(ValueFromPipelineByPropertyName=$true)]
    [System.Management.Automation.PSCredential]
    ${Credential}
)

begin {
    $wrappedCmd = $ExecutionContext.Invoke

In [11]:
# Alternative syntax to retrieve function definition
${Function:mkdir} | Out-Default


<#
.FORWARDHELPTARGETNAME New-Item
.FORWARDHELPCATEGORY Cmdlet
#>

[CmdletBinding(DefaultParameterSetName='pathSet',
    SupportsShouldProcess=$true,
    SupportsTransactions=$true,
    ConfirmImpact='Medium')]
    [OutputType([System.IO.DirectoryInfo])]
param(
    [Parameter(ParameterSetName='nameSet', Position=0, ValueFromPipelineByPropertyName=$true)]
    [Parameter(ParameterSetName='pathSet', Mandatory=$true, Position=0, ValueFromPipelineByPropertyName=$true)]
    [System.String[]]
    ${Path},

    [Parameter(ParameterSetName='nameSet', Mandatory=$true, ValueFromPipelineByPropertyName=$true)]
    [AllowNull()]
    [AllowEmptyString()]
    [System.String]
    ${Name},

    [Parameter(ValueFromPipeline=$true, ValueFromPipelineByPropertyName=$true)]
    [System.Object]
    ${Value},

    [Switch]
    ${Force},

    [Parameter(ValueFromPipelineByPropertyName=$true)]
    [System.Management.Automation.PSCredential]
    ${Credential}
)

begin {
    $wrappedCmd = $ExecutionContext.Invoke

#### Function Usage

`Global:<Function-Name>`  # makes a function available to the global scope

In [12]:
# List all acceptable PowerShell verb parts
Get-Verb | Select-Object Verb, Group, Description | Format-Table -AutoSize | Out-Default


Verb        Group          Description
----        -----          -----------
Add         Common         Adds a resource to a container, or attaches an item to another item
Clear       Common         Removes all the resources from a container but does not delete the cont…
Close       Common         Changes the state of a resource to make it inaccessible, unavailable, o…
Copy        Common         Copies a resource to another name or to another container
Enter       Common         Specifies an action that allows the user to move into a resource
Exit        Common         Sets the current environment or context to the most recently used conte…
Find        Common         Looks for an object in a container that is unknown, implied, optional, …
Format      Common         Arranges objects in a specified form or layout
Get         Common         Specifies an action that retrieves a resource
Hide        Common         Makes a resource undetectable
Join        Common         Combines resources

In [13]:
# When entering null parameter values
[Type]::Missing | Out-Default

System.Reflection.Missing
